# Week 10 Hands-On Lab — Probing Three Trustworthy-AI Pillars

**ESP3201 · formative hands-on lab.** Runs on free-tier Google Colab with a **CPU runtime** (no GPU, no API keys).

A "trustworthy AI" report that passes six pillar checks is a claim, not a proof. This lab gives you
real, runnable evidence for three of the pillars -- fairness, explainability, and privacy -- on toy
data you generate and inspect yourself. The graded Week 10 mini-assignment then asks you to apply
the same recomputation skill to a real (if fictional) audit report and its own data.

**Note:** the data in this lab is a separate illustrative example, not the data used in the graded
mini-assignment.

## Tasks

1. Recompute a fairness headline number from raw per-cell data and find the subgroup it hides.
2. Compare a model's local explanation at one instance against its actual global behavior.
3. Measure a real membership-inference confidence gap on a small, overfit model.
4. Fill the worksheet.

## Setup

In [ ]:
import sys, os

COURSE_REPO_URL = "https://github.com/bingquan/esp3201-public.git"

try:
    import sklearn
except ModuleNotFoundError:
    os.system("pip install -q scikit-learn")
try:
    import trustworthy_ai_lab
except ModuleNotFoundError:
    cand = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
    if os.path.exists(os.path.join(cand, "trustworthy_ai_lab.py")):
        sys.path.insert(0, cand)
    elif COURSE_REPO_URL:
        os.system(f"git clone -q {COURSE_REPO_URL} course_repo")
        sys.path.insert(0, "course_repo/mini_assignments/week10_trustworthy_ai_probe/starter")
    else:
        raise ModuleNotFoundError("trustworthy_ai_lab.py not found. On Colab set COURSE_REPO_URL.")
    import trustworthy_ai_lab
from trustworthy_ai_lab import (load_subgroup_csv, weighted_accuracy, per_cell_accuracy, worst_cell,
                                make_explainability_demo_data, train_explainability_demo_model,
                                local_explanation, global_feature_weight,
                                make_membership_inference_demo_data,
                                train_membership_inference_demo_model, membership_confidence_gap)
import numpy as np
print("trustworthy_ai_lab loaded.")

## Part A — Fairness: recompute the headline, find the hidden cell

A report can quote one weighted-average accuracy number and never show the per-condition breakdown.
Below is `fairness_probe_data.csv`: detection results for a toy person-detector across
indoor/outdoor and bright/dim conditions, split by an `adult`/`child` subgroup.

In [ ]:
rows = load_subgroup_csv("fairness_probe_data.csv")
for r in rows:
    print(r)

**First**, compute the single weighted-average accuracy a report would quote (this is exactly
`total_correct / total_samples` across every row -- write it yourself before running the next cell).

In [ ]:
headline = weighted_accuracy(rows)
print(f"weighted overall accuracy: {headline:.4f}")

**Now** break it down by cell, and find the worst one. Does the headline number tell you this
cell exists?

In [ ]:
for cell in per_cell_accuracy(rows):
    print(f"{cell['condition']:8s} {cell['lighting']:6s} {cell['subgroup']:6s} "
          f"n={cell['n_samples']:4d}  accuracy={cell['accuracy']:.3f}")

print()
print("WORST CELL:", worst_cell(rows))

### Checkpoint A — what does the headline hide?

In your own words: how far below the headline accuracy is the worst cell, and is that gap large enough that you'd want to know about it before deploying this detector outdoors, in the dark, on children? What is the headline number actively *not telling you*?

## Part B — Explainability: a local explanation is not the whole model

We'll train a small classifier where the label truly depends on `feature_a`. `feature_b` is mostly
noise -- **except** in a narrow band of `feature_a` values, where it happens to correlate with the
label by construction. This mimics a real failure mode: a feature that is spuriously predictive in
one region of the data, which a *local* explanation method can mistake for the model's real
reasoning.

In [ ]:
X, y = make_explainability_demo_data(seed=0)
model = train_explainability_demo_model(X, y)

global_weights = global_feature_weight(model)
print("GLOBAL model weights (what the model actually relies on, on average):")
print(f"  feature_a: {global_weights['feature_a']:.3f}")
print(f"  feature_b: {global_weights['feature_b']:.3f}")

Now pick an instance right in the middle of that narrow band (where `feature_a` is close to its decision boundary) and generate a **local** explanation for it.

In [ ]:
idx = int(np.argmin(np.abs(X[:, 0])))
x_row = X[idx]
print(f"instance: feature_a={x_row[0]:.3f}  feature_b={x_row[1]:.3f}  true label={y[idx]}")

explanation = local_explanation(model, x_row, X)
print("model\'s predicted probability of class 1 at this instance:", explanation["base_prob"])
print("LOCAL importance at this instance:")
print(f"  feature_a: {explanation['local_importance']['feature_a']:.3f}")
print(f"  feature_b: {explanation['local_importance']['feature_b']:.3f}")

### Checkpoint B — local versus global

Which feature does the GLOBAL weight say the model relies on? Which feature does the LOCAL explanation say mattered most *at this one instance*? If these disagree, what would a stakeholder wrongly conclude from reading only the local explanation for this instance? Is the local explanation *wrong*, or is it correct but insufficient as a claim about the model overall?

## Part C — Privacy: can you tell if a point was in the training set?

Train a small, high-capacity, unregularized model on a **very small** training set. A real
membership-inference attack asks: given a model and a data point, can you tell whether that point
was used to train it? One simple signal: the model's own confidence.

In [ ]:
X_train, y_train, X_test, y_test = make_membership_inference_demo_data(seed=0)
model = train_membership_inference_demo_model(X_train, y_train)

print(f"train accuracy: {model.score(X_train, y_train):.3f}")
print(f"test accuracy:  {model.score(X_test, y_test):.3f}")

gap = membership_confidence_gap(model, X_train, X_test)
print()
print(f"mean confidence on TRAINING points: {gap['mean_train_confidence']:.4f}")
print(f"mean confidence on HELD-OUT points: {gap['mean_test_confidence']:.4f}")
print(f"confidence gap:                    {gap['gap']:.4f}")

### Checkpoint C — what does the gap prove?

Compare the train/test *accuracy* gap with the train/test *confidence* gap. If someone handed you a single data point and asked "was this in the training set?", could you make a real guess using only the model's confidence? What does this imply about publishing a model (even without publishing its training data)?

## Worksheet (your deliverable)

### 1. Fairness

- Headline (weighted) accuracy:
- Worst cell (name it) and its accuracy:
- Gap between headline and worst cell:

### 2. Explainability

- Global weight ranking (which feature matters more overall):
- Local importance ranking at your chosen instance:
- Do they agree? If not, what is the risk of trusting only the local explanation?

### 3. Privacy

- Train accuracy / test accuracy:
- Train confidence / test confidence / gap:
- Could you use this gap to guess membership on a new point? Explain briefly.

### 4. Connect to the six pillars

Pick ONE of fairness, explainability, or privacy above and state, in one sentence, the evaluation
question this lab shows you cannot answer by reading a report's conclusion alone -- only by running
the numbers yourself.

## AI-Agent Usage Disclosure

State:

- which tools you used
- what they helped produce
- what you verified or rewrote yourself
- one specific thing you did not trust without checking